In [ ]:
import os,sys
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scanpy import AnnData
import mudata
from scipy.sparse import csr_matrix
import warnings
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import harmonypy as hm

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
sc.settings.verbosity = 3
warnings.filterwarnings("ignore")
plt.rcParams['pdf.fonttype']=42

In [ ]:
bgs5 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260108_bgs5_filtered_snm3_3T_L3.h5ad")

#3T --> bgs5_filtered hat was made just now
bgs5

In [ ]:
#resegmented

bgs4 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260107_bgs4_reseg_snm3_2T_L3.h5ad")

bgs4

In [ ]:
#change bgs4 '?' cell type to 'eMGE'

label_col = 'adjusted_L3_transferred'

bgs4.obs[label_col] = (
    bgs4.obs[label_col]
    .replace({'?': 'eMGE'})
)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Two separate heatmaps: marker gene expression across cell types
# - Consistent color scale between top and bottom panels
# - Z-scored pseudobulk expression (NO CLIPPING)
# - Cell counts shown on the right

# 1. Explicitly define genes to show (in desired order)
custom_gene_order = [
    'DRD1',      # Direct MSN marker
    'DRD2',      # Indirect MSN marker

    'BACH2',     # Transcription factor (DRD1 subtype)
    'EPHA4',     # Eph receptor (DRD1/DRD2 subtype)
    'PENK',      # Enkephalin - classic D2-MSN marker (indirect pathway)
    'GAD1',      # GABAergic interneuron marker
    'NKX2-1',
    'EGFR',
    'PAX6',
    #'PDGFRA',
    'OLIG1',
    #'SOX2',
    #'NES',

    'OLIG2',     # Oligodendrocyte lineage
    'SOX10',     # Oligodendrocyte lineage
    'PDGFRA',     # Strong OPC and pericyte marker (careful overlap with PC)
    'MBP',        # Myelin basic protein - mature oligodendrocyte marker

    'ALDH1L1',   # Astrocyte marker
    'GFAP',      # Astrocyte marker
    'AQP4',      # Astrocyte marker

    'ESAM',      # Endothelial marker
    'FN1',       # Endothelial / pericyte
    #'LUM',        # Lumican - top VLMC/fibroblast-like marker
    'DCN',        # Decorin - VLMC marker

    'P2RY12',    # Microglia marker
    'CX3CR1',     # Fractalkine receptor - microglia marker (complements P2RY12)
    #'SLC17A7',    # VGLUT1 - excitatory neuron marker (useful for any glutamatergic contamination or comparison)

]

# 2. Explicitly define cell types to KEEP for EACH dataset
# Edit these lists independently - only types in the list will appear in that heatmap
cell_types_bgs4 = [          # 2T  GW24
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    #'PAX6-TCx',
    'Astro',

    'EC',
    'PC',
    #'VLMC',
    'MGC-1',
    # 'uncertain',  # excluded
]

cell_types_bgs5 = [          # 3T  GW35
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',

    'GPC',
    'ODC',
    'OPC',
    'Astro',

    #'PVALB',
    #'SST',
    'EC',
    'PC',
    #'VLMC',
    'MGC-1',
    # 'uncertain',  # excluded
]

# 3. Setup
label_col = 'adjusted_L3_transferred'

bgs4_f = bgs4.copy()
bgs5_f = bgs5.copy()

# 4. Restrict to desired cell types
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(cell_types_bgs4)].copy()
bgs5_f = bgs5_f[bgs5_f.obs[label_col].isin(cell_types_bgs5)].copy()

# 5. Pseudobulk z-score function (NO CLIPPING)
def pseudobulk_zscore_fine(adata, genes, groupby):
    genes_use = [g for g in genes if g in adata.var_names]
    print(f"{adata.uns.get('name','adata')} - using {len(genes_use)}/{len(genes)} genes")

    if 'counts_downsampled' in adata.layers:
        X = adata[:, genes_use].layers['counts_downsampled']
    else:
        X = adata[:, genes_use].X

    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values

    pb = df.groupby(groupby).mean()
    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    pb_z = pb_z.fillna(0)

    return pb_z, genes_use

# 6. Compute pseudobulk matrices
pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order, label_col)
pb5, genes5 = pseudobulk_zscore_fine(bgs5_f, custom_gene_order, label_col)

shared_genes = [g for g in custom_gene_order if g in genes4 and g in genes5]
pb4 = pb4.loc[:, shared_genes]
pb5 = pb5.loc[:, shared_genes]

pb4 = pb4.reindex(cell_types_bgs4)
pb5 = pb5.reindex(cell_types_bgs5)

# 7. Compute GLOBAL color scale (shared)
global_min = min(pb4.min().min(), pb5.min().min())
global_max = max(pb4.max().max(), pb5.max().max())

# Symmetric scaling around zero (recommended for z-scores)
abs_max = max(abs(global_min), abs(global_max))
vmin, vmax = -abs_max, abs_max

print(f"Global color scale: vmin={vmin:.2f}, vmax={vmax:.2f}")

# 8. Cell counts
counts4 = (
    bgs4_f.obs[label_col]
    .value_counts()
    .reindex(pb4.index)
    .fillna(0)
    .astype(int)
)

counts5 = (
    bgs5_f.obs[label_col]
    .value_counts()
    .reindex(pb5.index)
    .fillna(0)
    .astype(int)
)

# 9. Plotting
fig = plt.figure(figsize=(14, (len(pb4) + len(pb5)) * 0.32 + 6))
gs = fig.add_gridspec(2, 1, hspace=0.2, height_ratios=[len(pb4), len(pb5)])

# Top: 2T  GW24
ax1 = fig.add_subplot(gs[0])
g1 = sns.heatmap(
    pb4,
    cmap='vlag',
    center=0,
    vmin=vmin,
    vmax=vmax,
    linewidths=0.5,
    linecolor='lightgray',
    cbar_kws={"orientation": "horizontal", "pad": 0.35, "shrink": 0.6},

    ax=ax1
)

ax1.set_title(
    "2T  GW24 Label Transfer - Selected Cell Types",
    fontsize=15,
    pad=12
)
ax1.set_ylabel("Cell Type", fontsize=14)
ax1.set_xlabel("Genes", fontsize=14)
ax1.tick_params(axis='x', rotation=90, labelsize=11)

cbar1 = g1.collections[0].colorbar
cbar1.set_label("z-score", fontsize=11)

for i, n in enumerate(counts4):
    ax1.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}",
             va='center', ha='left', fontsize=13)
ax1.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

# Bottom: 3T  GW35
ax2 = fig.add_subplot(gs[1])
g2 = sns.heatmap(
    pb5,
    cmap='vlag',
    center=0,
    vmin=vmin,
    vmax=vmax,
    linewidths=0.5,
    linecolor='lightgray',
    cbar_kws={"orientation": "horizontal", "pad": 0.2, "shrink": 0.6},
    ax=ax2
)

ax2.set_title(
    "3T  GW35 Label Transfer - Selected Cell Types",
    fontsize=15,
    pad=12
)
ax2.set_ylabel("Cell Type", fontsize=14)
ax2.set_xlabel("Genes", fontsize=14)
ax2.tick_params(axis='x', rotation=90, labelsize=11)

cbar2 = g2.collections[0].colorbar
cbar2.set_label("z-score", fontsize=11)

for i, n in enumerate(counts5):
    ax2.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}",
             va='center', ha='left', fontsize=13)
#ax2.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

# 10. Final layout
plt.suptitle(
    "\nMarker Gene Expression Across Selected Cell Types\n"
    "\n2T  GW24 (top) vs 3T  GW35 (bottom)",
    fontsize=17,
    y=0.985
)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import scipy.sparse as sp
from pathlib import Path

# Matplotlib settings for Illustrator-friendly SVG
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'   # keep text editable in SVG

# Output directory
outdir = Path("20260114 - figs")
outdir.mkdir(exist_ok=True)

# Two separate heatmaps: marker gene expression across cell types
# - Consistent color scale between panels
# - Z-scored pseudobulk expression (NO clipping)
# - Cell counts shown on the right

# 1. Explicit gene order
custom_gene_order = [
    'DRD1', 'DRD2',
    'BACH2', 'EPHA4', 'PENK', 'GAD1',
    'NKX2-1', 'EGFR', 'PAX6', 'OLIG1',
    'OLIG2', 'SOX10', 'PDGFRA', 'MBP',
    'ALDH1L1', 'GFAP', 'AQP4',
    'ESAM', 'FN1', 'DCN',
    'P2RY12', 'CX3CR1',
]

# 2. Cell types to keep
cell_types_bgs4 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4',
    'eMSN', 'eMGE',
    'GPC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1',
]

cell_types_bgs5 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC', 'ODC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1',
]

# 3. Setup
label_col = 'adjusted_L3_transferred'
bgs4_f = bgs4[bgs4.obs[label_col].isin(cell_types_bgs4)].copy()
bgs5_f = bgs5[bgs5.obs[label_col].isin(cell_types_bgs5)].copy()

# 4. Pseudobulk z-score (NO clipping)
def pseudobulk_zscore_fine(adata, genes, groupby):
    genes_use = [g for g in genes if g in adata.var_names]

    X = (
        adata[:, genes_use].layers['counts_downsampled']
        if 'counts_downsampled' in adata.layers
        else adata[:, genes_use].X
    )
    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values

    pb = df.groupby(groupby).mean()
    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    return pb_z.fillna(0), genes_use

pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order, label_col)
pb5, genes5 = pseudobulk_zscore_fine(bgs5_f, custom_gene_order, label_col)

shared_genes = [g for g in custom_gene_order if g in genes4 and g in genes5]
pb4 = pb4.loc[cell_types_bgs4, shared_genes]
pb5 = pb5.loc[cell_types_bgs5, shared_genes]

# 5. Shared color scale
abs_max = max(abs(pb4.values).max(), abs(pb5.values).max())
vmin, vmax = -abs_max, abs_max

# 6. Cell counts
counts4 = bgs4_f.obs[label_col].value_counts().reindex(pb4.index).astype(int)
counts5 = bgs5_f.obs[label_col].value_counts().reindex(pb5.index).astype(int)

# 7. Plotting
fig = plt.figure(figsize=(14, (len(pb4) + len(pb5)) * 0.32 + 9))
gs = fig.add_gridspec(2, 1, hspace=0.2, height_ratios=[len(pb4), len(pb5)])

# Top panel (GW24)
ax1 = fig.add_subplot(gs[0])
g1 = sns.heatmap(
    pb4, cmap='vlag', center=0,
    vmin=vmin, vmax=vmax,
    linewidths=0.5, linecolor='lightgray',
    cbar_kws={"orientation": "horizontal", "pad": 0.22, "shrink": 0.6},
    ax=ax1
)

ax1.set_title("2T  GW24 Label Transfer - Selected Cell Types", fontsize=15, pad=12)
ax1.set_ylabel("Cell Type\n", fontsize=14)
ax1.set_xlabel("Genes", fontsize=14)
ax1.tick_params(axis='x', rotation=90, labelsize=13)
ax1.tick_params(axis='y', labelsize=13)

for i, n in enumerate(counts4):
    ax1.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}",
             va='center', ha='left', fontsize=13)

# Bottom panel (GW35)
ax2 = fig.add_subplot(gs[1])
g2 = sns.heatmap(
    pb5, cmap='vlag', center=0,
    vmin=vmin, vmax=vmax,
    linewidths=0.5, linecolor='lightgray',
    cbar_kws={"orientation": "horizontal", "pad": 0.20, "shrink": 0.6},
    ax=ax2
)

ax2.set_title("3T  GW35 Label Transfer - Selected Cell Types", fontsize=15, pad=12)
ax2.set_ylabel("Cell Type\n", fontsize=14)
ax2.set_xlabel("Genes", fontsize=14)
ax2.tick_params(axis='x', rotation=90, labelsize=13)
ax2.tick_params(axis='y', labelsize=13)

for i, n in enumerate(counts5):
    ax2.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}",
             va='center', ha='left', fontsize=13)

# 8. Final layout + save
plt.suptitle(
    "\nMarker Gene Expression Across Selected Cell Types",
    fontsize=17,
    y=0.985
)

fig.savefig(
    outdir / "bgs4_bgs5_marker_heatmaps.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import scipy.sparse as sp
from pathlib import Path

# Matplotlib settings for Illustrator-friendly PDF
# - editable text (TrueType fonts embedded)
# - Arial font
mpl.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',   # fine to keep; not used for PDF
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
})

# Output: current notebook working directory
outpath = Path.cwd() / "20260115_bgs4_bgs5_marker_heatmaps.pdf"

# Two separate heatmaps: marker gene expression across cell types
# - Consistent color scale between panels
# - Z-scored pseudobulk expression (NO clipping)
# - Cell counts shown on the right

# 1. Explicit gene order
custom_gene_order = [
    'DRD1', 'DRD2',
    'BACH2', 'EPHA4', 'PENK', 'GAD1',
    'NKX2-1', 'EGFR', 'PAX6', 'OLIG1',
    'OLIG2', 'SOX10', 'PDGFRA', 'MBP',
    'ALDH1L1', 'GFAP', 'AQP4',
    'ESAM', 'FN1', 'DCN',
    'P2RY12', 'CX3CR1',
]

# 2. Cell types to keep
cell_types_bgs4 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4',
    'eMSN', 'eMGE',
    'GPC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1',
]

cell_types_bgs5 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC', 'ODC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1',
]

# 3. Setup
label_col = 'adjusted_L3_transferred'
bgs4_f = bgs4[bgs4.obs[label_col].isin(cell_types_bgs4)].copy()
bgs5_f = bgs5[bgs5.obs[label_col].isin(cell_types_bgs5)].copy()

# 4. Pseudobulk z-score (NO clipping)
def pseudobulk_zscore_fine(adata, genes, groupby):
    genes_use = [g for g in genes if g in adata.var_names]

    X = (
        adata[:, genes_use].layers['counts_downsampled']
        if 'counts_downsampled' in adata.layers
        else adata[:, genes_use].X
    )
    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values

    pb = df.groupby(groupby).mean()
    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    return pb_z.fillna(0), genes_use

pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order, label_col)
pb5, genes5 = pseudobulk_zscore_fine(bgs5_f, custom_gene_order, label_col)

shared_genes = [g for g in custom_gene_order if g in genes4 and g in genes5]
pb4 = pb4.loc[cell_types_bgs4, shared_genes]
pb5 = pb5.loc[cell_types_bgs5, shared_genes]

# 5. Shared color scale
abs_max = max(abs(pb4.values).max(), abs(pb5.values).max())
vmin, vmax = -abs_max, abs_max

# 6. Cell counts
counts4 = bgs4_f.obs[label_col].value_counts().reindex(pb4.index).astype(int)
counts5 = bgs5_f.obs[label_col].value_counts().reindex(pb5.index).astype(int)

# 7. Plotting
fig = plt.figure(figsize=(14, (len(pb4) + len(pb5)) * 0.32 + 9))
gs = fig.add_gridspec(2, 1, hspace=0.2, height_ratios=[len(pb4), len(pb5)])

# Top panel (GW24)
ax1 = fig.add_subplot(gs[0])
sns.heatmap(
    pb4, cmap='vlag', center=0,
    vmin=vmin, vmax=vmax,
    linewidths=0.5, linecolor='lightgray',
    cbar_kws={"orientation": "horizontal", "pad": 0.22, "shrink": 0.6},
    ax=ax1
)

ax1.set_title("2T  GW24 Label Transfer - Selected Cell Types", fontsize=15, pad=12)
ax1.set_ylabel("Cell Type\n", fontsize=14)
ax1.set_xlabel("Genes", fontsize=14)
ax1.tick_params(axis='x', rotation=90, labelsize=13)
ax1.tick_params(axis='y', labelsize=13)

for i, n in enumerate(counts4):
    ax1.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}",
             va='center', ha='left', fontsize=13)

# Bottom panel (GW35)
ax2 = fig.add_subplot(gs[1])
sns.heatmap(
    pb5, cmap='vlag', center=0,
    vmin=vmin, vmax=vmax,
    linewidths=0.5, linecolor='lightgray',
    cbar_kws={"orientation": "horizontal", "pad": 0.20, "shrink": 0.6},
    ax=ax2
)

ax2.set_title("3T  GW35 Label Transfer - Selected Cell Types", fontsize=15, pad=12)
ax2.set_ylabel("Cell Type\n", fontsize=14)
ax2.set_xlabel("Genes", fontsize=14)
ax2.tick_params(axis='x', rotation=90, labelsize=13)
ax2.tick_params(axis='y', labelsize=13)

for i, n in enumerate(counts5):
    ax2.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}",
             va='center', ha='left', fontsize=13)

# 8. Final layout + save to PDF (current directory)
plt.suptitle(
    "\nMarker Gene Expression Across Selected Cell Types",
    fontsize=17,
    y=0.985
)

fig.savefig(
    outpath,
    format="pdf",
    bbox_inches="tight"
)

plt.show()
print(f"Saved PDF to: {outpath}")

In [ ]:
bgs5 = sc.read_h5ad("/u/project/cluo/rayirfan/cosmx/analyses/basalganglia/D_snm3/bgs5_snm3_3T_L2_L3.h5ad")

#3T --> bgs5

bgs5

In [ ]:
bgs4 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260107_bgs4_reseg_snm3_2T_L3.h5ad")

#2T --> bgs4 LGE and striatum only

bgs4

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Two separate heatmaps: marker gene expression across cell types
# - Titles: "2T  GW24 (bgs4)" and "3T  GW35 (bgs5)"
# - Z-score clipped to [-2, 2]
# - Cell counts shown on the right
# - Fully customizable: genes + separate cell type lists for each dataset

# 1. Explicitly define genes to show (in desired order)
custom_gene_order = [
    'DRD1',      # Direct MSN marker
    'DRD2',      # Indirect MSN marker

    'BACH2',     # Transcription factor (DRD1 subtype)
    'EPHA4',     # Eph receptor (DRD1/DRD2 subtype)
    'PENK',      # Enkephalin - classic D2-MSN marker (indirect pathway)
    'GAD1',      # GABAergic interneuron marker

    'PAX6',
    #'PDGFRA',
    'OLIG1',
    #'SOX2',
    #'NES',

    'OLIG2',     # Oligodendrocyte lineage
    'SOX10',     # Oligodendrocyte lineage
    'PDGFRA',     # Strong OPC and pericyte marker (careful overlap with PC)
    'MBP',        # Myelin basic protein - mature oligodendrocyte marker

    'ALDH1L1',   # Astrocyte marker
    'GFAP',      # Astrocyte marker
    'AQP4',      # Astrocyte marker

    'ESAM',      # Endothelial marker
    'FN1',       # Endothelial / pericyte
    'LUM',        # Lumican - top VLMC/fibroblast-like marker
    'DCN',        # Decorin - VLMC marker

    'P2RY12',    # Microglia marker
    'CX3CR1',     # Fractalkine receptor - microglia marker (complements P2RY12)
    #'SLC17A7',    # VGLUT1 - excitatory neuron marker (useful for any glutamatergic contamination or comparison)

]

# 2. Explicitly define cell types to KEEP for EACH dataset
# Edit these lists independently - only types in the list will appear in that heatmap
cell_types_bgs4 = [          # 2T  GW24
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',

    'GPC',
    'OPC',
    #'PAX6-TCx',
    'Astro',

    'EC',
    'PC',
    'MGC-1',
    # 'uncertain',  # excluded
]

cell_types_bgs5 = [          # 3T  GW35
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',

    'GPC',
    'ODC',
    'OPC',
    'Astro',

    #'PVALB',
    #'SST',
    'EC',
    'PC',
    'VLMC',
    'MGC-1',
    # 'uncertain',  # excluded
]

# 3. Filtering and area-based QC
label_col = 'adjusted_L3_transferred'
min_area_um2 = 10
um_per_pixel = 0.12028

bgs4_f = bgs4.copy()
bgs5_f = bgs5.copy()

# for adata in [bgs4_f, bgs5_f]:
#     nuc_area = adata.obs['NucArea'] * (um_per_pixel ** 2)
#     cell_area = adata.obs['Area'] * (um_per_pixel ** 2)
#     mask = (nuc_area >= min_area_um2) & (cell_area >= min_area_um2)
#     adata = adata[mask].copy()

# 4. Restrict to desired cell types for each dataset separately
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(cell_types_bgs4)].copy()
bgs5_f = bgs5_f[bgs5_f.obs[label_col].isin(cell_types_bgs5)].copy()

# 5. Pseudobulk z-score function
def pseudobulk_zscore_fine(adata, genes, groupby=label_col):
    genes_use = [g for g in genes if g in adata.var_names]
    print(f"{adata.uns.get('name','adata')} - using {len(genes_use)}/{len(genes)} genes")

    if 'counts_downsampled' in adata.layers:
        X = adata[:, genes_use].layers['counts_downsampled']
    else:
        X = adata[:, genes_use].X

    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values
    pb = df.groupby(groupby).mean()

    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    pb_z = pb_z.fillna(0)
    pb_z = pb_z.clip(lower=-2, upper=2)

    return pb_z, genes_use

# 6. Compute pseudobulk for each dataset
pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order)
pb5, genes5 = pseudobulk_zscore_fine(bgs5_f, custom_gene_order)

# Use only genes present in BOTH datasets (for visual alignment)
shared_genes = [g for g in custom_gene_order if g in genes4 and g in genes5]
pb4 = pb4.loc[:, shared_genes]
pb5 = pb5.loc[:, shared_genes]

# 6b. Reorder cell types in pseudobulk according to custom order
pb4 = pb4.reindex(cell_types_bgs4)
pb5 = pb5.reindex(cell_types_bgs5)

# 7. Cell counts
counts4 = bgs4_f.obs[label_col].value_counts().reindex(pb4.index).fillna(0).astype(int)
counts5 = bgs5_f.obs[label_col].value_counts().reindex(pb5.index).fillna(0).astype(int)

# 8. Plotting
fig = plt.figure(figsize=(14, (len(pb4) + len(pb5)) * 0.32 + 6))
gs = fig.add_gridspec(2, 1, hspace=0.2, height_ratios=[len(pb4), len(pb5)])

# Top: 2T  GW24
ax1 = fig.add_subplot(gs[0])
g1 = sns.heatmap(pb4, cmap='vlag', vmin=-2, vmax=2, center=0,
                 linewidths=0.5, linecolor='lightgray',
                 cbar_kws={"orientation": "horizontal", "pad": 0.2, "shrink": 0.6},  # <- increase pad
                 ax=ax1)
ax1.set_title("2T  GW24 (bgs4) - Selected Cell Types (pseudobulk z-scored, clipped [-2, 2])",
              fontsize=15, pad=12)
ax1.set_ylabel("Cell Type", fontsize=13)
ax1.set_xlabel("")
ax1.tick_params(axis='x', rotation=90, labelsize=11)
cbar1 = g1.collections[0].colorbar
cbar1.set_label('z-score', fontsize=11)

# Cell counts (right side)
for i, n in enumerate(counts4):
    ax1.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}", va='center', ha='left', fontsize=9)
ax1.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

# Bottom: 3T  GW35
ax2 = fig.add_subplot(gs[1])
g2 = sns.heatmap(pb5, cmap='vlag', vmin=-2, vmax=2, center=0,
                 linewidths=0.5, linecolor='lightgray',
                 cbar_kws={"orientation": "horizontal", "pad": 0.2, "shrink": 0.6},  # <- increase pad
                 ax=ax2)
ax2.set_title("3T  GW35 (bgs5) - Selected Cell Types (pseudobulk z-scored, clipped [-2, 2])",
              fontsize=15, pad=12)
ax2.set_ylabel("Cell Type", fontsize=13)
ax2.set_xlabel("Genes", fontsize=13)
ax2.tick_params(axis='x', rotation=90, labelsize=11)
cbar2 = g2.collections[0].colorbar
cbar2.set_label('z-score', fontsize=11)

# Cell counts (right side)
for i, n in enumerate(counts5):
    ax2.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}", va='center', ha='left', fontsize=9)
ax2.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

plt.suptitle("\nMarker Gene Expression Across Selected Cell Types\n2T  GW24 (top) vs 3T  GW35 (bottom)",
             fontsize=17, y=0.985)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 1. Define genes to show
custom_gene_order = [
    'DRD1', 'DRD2', 'BACH2', 'EPHA4', 'PENK', 'GAD1',
    'PAX6', 'OLIG1', 'OLIG2', 'SOX10', 'PDGFRA', 'MBP',
    'ALDH1L1', 'GFAP', 'AQP4', 'ESAM', 'FN1', 'LUM', 'DCN',
    'P2RY12', 'CX3CR1'
]

# 2. Define custom cell type order
cell_types_bgs4 = [
    'DRD1-BACH2','DRD1-EPHA4','DRD1-eccentric-CASZ1','DRD2-BACH2','DRD2-EPHA4','eMSN',
    'GPC','OPC','Astro','EC','PC','MGC-1'
]

cell_types_bgs5 = [
    'DRD1-BACH2','DRD1-EPHA4','DRD1-eccentric-CASZ1','DRD2-BACH2','DRD2-EPHA4','DRD2-eccentric-CASZ1',
    'CABLES1','GPC','ODC','OPC','Astro','EC','PC','VLMC','MGC-1'
]

# 3. Filter genes actually present in the dataset
available_genes_bgs4 = [g for g in custom_gene_order if g in bgs4.var_names]
available_genes_bgs5 = [g for g in custom_gene_order if g in bgs5.var_names]

# 4. Prepare copies for plotting
adata_bgs4_plot = bgs4[:, available_genes_bgs4].copy()
adata_bgs5_plot = bgs5[:, available_genes_bgs5].copy()

# Use raw counts if available
if 'counts' in adata_bgs4_plot.layers:
    adata_bgs4_plot.X = adata_bgs4_plot.layers['counts'].copy()
if 'counts' in adata_bgs5_plot.layers:
    adata_bgs5_plot.X = adata_bgs5_plot.layers['counts'].copy()

# 5. Normalize and log1p
sc.pp.normalize_total(adata_bgs4_plot, target_sum=1e4)
sc.pp.log1p(adata_bgs4_plot)

sc.pp.normalize_total(adata_bgs5_plot, target_sum=1e4)
sc.pp.log1p(adata_bgs5_plot)

# 6. Enforce custom cell type order
adata_bgs4_plot.obs['adjusted_L3_transferred'] = pd.Categorical(
    adata_bgs4_plot.obs['adjusted_L3_transferred'],
    categories=cell_types_bgs4,
    ordered=True
)

adata_bgs5_plot.obs['adjusted_L3_transferred'] = pd.Categorical(
    adata_bgs5_plot.obs['adjusted_L3_transferred'],
    categories=cell_types_bgs5,
    ordered=True
)

# 7. Plot dotplot for bgs4 (2T  GW24)
sc.pl.dotplot(
    adata_bgs4_plot,
    var_names=available_genes_bgs4,
    groupby='adjusted_L3_transferred',
    use_raw=False,
    cmap='Reds',
    dot_max=0.8,
    dot_min=0.05,
    smallest_dot=20,
    standard_scale='var',     # normalize expression per gene
    title='2T  GW24 (bgs4) - Marker Gene Expression',
    figsize=(20, 8),
    dendrogram=False,
    show=False
)

# 8. Plot dotplot for bgs5 (3T  GW35)
sc.pl.dotplot(
    adata_bgs5_plot,
    var_names=available_genes_bgs5,
    groupby='adjusted_L3_transferred',
    use_raw=False,
    cmap='Reds',
    dot_max=0.8,
    dot_min=0.05,
    smallest_dot=20,
    standard_scale='var',
    title='3T  GW35 (bgs5) - Marker Gene Expression',
    figsize=(20, 10),
    dendrogram=False,
    show=False
)

plt.tight_layout()
plt.show()